# 🧠 Phase 5: YOLO v2 Multi-Class Model Trainer (4 Core ICT Classes)

### 🎯 Phase 5 Target:
Train a dedicated Computer Vision model to detect **4 Fundamental ICT PD Arrays** simultaneously:
- `Class 0`: **Bullish FVG** (BISI - Buy-Side Imbalance)
- `Class 1`: **Bearish FVG** (SIBI - Sell-Side Imbalance)
- `Class 2`: **Bullish Order Block (OB)** (Last down-close candle before displacement + FVG)
- `Class 3`: **Bearish Order Block (OB)** (Last up-close candle before displacement + FVG)

### ⚡ Hardware:
Set Runtime to **T4 GPU** (`Runtime > Change runtime type > T4 GPU > Save`), then click **Runtime > Run all**.

### Step 1: Install Dependencies & Import Libraries

In [ ]:
!pip install -q ultralytics mplfinance huggingface_hub pyarrow

import os, shutil, glob, zoneinfo
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import mplfinance as mpf
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

plt.ioff()
NY_TZ = zoneinfo.ZoneInfo("America/New_York")
print("✅ Setup ready for 4-Class ICT YOLO Training!")

### Step 2: Load Historical XAU/USD 15m Data (New York Time)

In [ ]:
REPO = "SpatialGhost/xauusd-gold-price-historical-data-2004-2025"
FILE = "XAU_15m_data.jsonl"
START_YEAR = 2023

print("⏳ Downloading Gold dataset...")
path = hf_hub_download(REPO, FILE, repo_type="dataset")

chunks = []
for c in pd.read_json(path, lines=True, chunksize=150000):
    c = c.rename(columns=lambda x: str(x).strip().capitalize())
    c["Date"] = pd.to_datetime(c["Date"], errors="coerce")
    c = c.dropna(subset=["Date"])
    c = c[c["Date"].dt.year >= START_YEAR]
    chunks.append(c)

df_raw = pd.concat(chunks)
if df_raw["Date"].dt.tz is None:
    df_raw["Date"] = df_raw["Date"].dt.tz_localize("UTC")
df_raw["Date_NY"] = df_raw["Date"].dt.tz_convert(NY_TZ)
df = df_raw.set_index("Date_NY").sort_index()
df = df[~df.index.duplicated()][["Open", "High", "Low", "Close", "Volume"]].dropna()

print(f"✅ Clean 15m Gold Data Ready: {len(df):,} candles!")
df.tail(3)

### Step 3: Mathematical 4-Class Auto-Labeler (FVGs + ICT Order Blocks)

In [ ]:
shutil.rmtree("ds_v2", ignore_errors=True)
for s in ["train", "val"]:
    os.makedirs(f"ds_v2/images/{s}", exist_ok=True)
    os.makedirs(f"ds_v2/labels/{s}", exist_ok=True)

W, STEP, DPI, MAX_IMAGES = 100, 50, 100, 1500

def find_4class_zones(w):
    hi, lo, op, cl = w.High.values, w.Low.values, w.Open.values, w.Close.values
    min_fvg = (w.High - w.Low).mean() * 0.45
    zones = []
    
    for i in range(2, len(w)):
        # 1. Bullish FVG (Class 0)
        if lo[i] - hi[i-2] > min_fvg:
            zones.append((0, i-2, hi[i-2], lo[i], min(i+10, W-1)))  # Class 0: Bull FVG
            # Find Bullish Order Block (Class 2): last down candle before displacement
            for ob_i in range(i-2, max(0, i-6), -1):
                if cl[ob_i] < op[ob_i]: # down candle
                    zones.append((2, ob_i, lo[ob_i], hi[ob_i], min(ob_i+12, W-1))) # Class 2: Bull OB
                    break
                    
        # 2. Bearish FVG (Class 1)
        if lo[i-2] - hi[i] > min_fvg:
            zones.append((1, i-2, hi[i], lo[i-2], min(i+10, W-1)))  # Class 1: Bear FVG
            # Find Bearish Order Block (Class 3): last up candle before displacement
            for ob_i in range(i-2, max(0, i-6), -1):
                if cl[ob_i] > op[ob_i]: # up candle
                    zones.append((3, ob_i, lo[ob_i], hi[ob_i], min(ob_i+12, W-1))) # Class 3: Bear OB
                    break
    return zones

starts = list(range(0, len(df) - W, STEP))[-MAX_IMAGES:]
split_at = int(len(starts) * 0.8)
saved = 0
print("⏳ Generating 4-Class Labeled Candlestick Dataset...")

for k, st in enumerate(starts):
    w = df.iloc[st:st+W]
    zones = find_4class_zones(w)
    if not zones:
        continue
    s = "train" if k < split_at else "val"
    fig, axes = mpf.plot(w, type="candle", style="charles", axisoff=True,
                         figsize=(8, 8), returnfig=True)
    fig.set_dpi(DPI)
    ax = axes[0]
    fig.canvas.draw()
    H = 8 * DPI
    lines = []
    for c, i0, zl, zh, i1 in zones:
        x0, y0 = ax.transData.transform((i0 - 0.5, zl))
        x1, y1 = ax.transData.transform((i1 + 0.5, zh))
        # Normalize to YOLO format (class, xc, yc, w, h)
        xc = (x0 + x1) / 2 / H
        yc = 1.0 - (y0 + y1) / 2 / H
        bw = abs(x1 - x0) / H
        bh = abs(y1 - y0) / H
        # clamp between 0 and 1
        xc, yc, bw, bh = max(0, min(1, xc)), max(0, min(1, yc)), max(0.01, min(1, bw)), max(0.01, min(1, bh))
        lines.append(f"{c} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
        
    fig.savefig(f"ds_v2/images/{s}/{k}.png", dpi=DPI)
    plt.close(fig)
    with open(f"ds_v2/labels/{s}/{k}.txt", "w") as f:
        f.write("\n".join(lines))
    saved += 1
    if saved % 250 == 0:
        print(f"  Progress: {saved} charts generated...")

yaml_content = """path: /content/ds_v2
train: images/train
val: images/val
names:
  0: bull_fvg
  1: bear_fvg
  2: bull_ob
  3: bear_ob
"""
with open("ds_v2/data.yaml", "w") as f:
    f.write(yaml_content)

print(f"✅ Dataset Generation Complete! Total labeled charts: {saved}")

### Step 4: Train YOLOv8 Multi-Class Vision Model (35 Epochs)

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("❌ GPU CONNECT NAHI HAI! Aapka code CPU par chal raha hai isliye 10 min lag rahe hain. Upar menu mein: Runtime > Change runtime type > T4 GPU select karke Save karo, fir Run all karo!")

print(f"⚡ GPU DETECTED: {torch.cuda.get_device_name(0)}! Superfast speed active.")
print("🚀 Training shuru ho rahi hai local fast memory par (~15-20 sec per epoch)...\n")
model = YOLO("yolov8n.pt")
results = model.train(
    data="/content/ds_v2/data.yaml",
    epochs=25,
    imgsz=640,
    batch=16,
    project="/content/runs",
    name="ict_v2_4class",
    exist_ok=True,
    plots=False
)

best_v2 = "/content/runs/ict_v2_4class/weights/best.pt"
print("\n🎉 Training Complete! BEST MODEL:", best_v2)

# Google Drive mein safe copy
try:
    from google.colab import drive
    drive.mount('/content/drive')
    D = '/content/drive/MyDrive/chart_model'
    os.makedirs(D, exist_ok=True)
    shutil.copy(best_v2, D + '/best_v2_4class.pt')
    print(f"💾 Model safely saved to Google Drive: {D}/best_v2_4class.pt")
except Exception as e:
    print("Drive copy warning:", e)

### Step 5: Evaluate Per-Class Precision, Recall & mAP50 Metrics

In [ ]:
print("=== 📊 PER-CLASS ACCURACY METRICS ===\n")
metrics = model.val()
class_names = ["bull_fvg", "bear_fvg", "bull_ob", "bear_ob"]

print(f"{'Class':<12} {'Precision':<12} {'Recall':<12} {'mAP50':<12}")
print("-" * 48)
for i, cname in enumerate(class_names):
    p = metrics.box.p[i] if i < len(metrics.box.p) else 0.0
    r = metrics.box.r[i] if i < len(metrics.box.r) else 0.0
    map50 = metrics.box.ap50[i] if i < len(metrics.box.ap50) else 0.0
    print(f"{cname:<12} {p:<12.3f} {r:<12.3f} {map50:<12.3f}")

print("-" * 48)
print(f"Overall All-Class mAP50: {metrics.box.map50:.3f}")

### Step 6: Test Visual Detection on Validation Charts

In [ ]:
import cv2
from google.colab.patches import cv2_imshow

val_imgs = sorted(glob.glob("/content/ds_v2/images/val/*.png"))[:3]
print("Validation samples with 4-Class Detections:")
for p in val_imgs:
    res = YOLO(best_v2)(p, conf=0.3, verbose=False)[0]
    cv2_imshow(res.plot())

### Step 7: Save to Google Drive & PC Download

In [ ]:
from google.colab import drive, files

try:
    drive.mount('/content/drive')
    D = '/content/drive/MyDrive/chart_model'
    os.makedirs(D, exist_ok=True)
    shutil.copy(best_v2, D + '/best_v2_4class.pt')
    print(f"✅ Successfully saved to Google Drive: {D}/best_v2_4class.pt")
except Exception as e:
    print("Drive mount warning:", e)

# Direct PC Download
print("📥 Downloading best_v2_4class.pt to PC/Phone...")
files.download(best_v2)